In [56]:
# Get the sqlite database path
#| echo: false

import pandas as pd
from great_tables import GT
from great_tables import html

import nhs_waiting_lists as nhs
from nhs_waiting_lists.constants import LARGE_ACUTE_PROVIDER_CODES, TOTAL_ONLY_TREATMENT_CODES


In [57]:
#| echo: false
#| output: false

## Retrieve all the provider rtt summary data

start_period = "2024-01"
end_period = "2024-12"

# Get the 23 large acute trust provider codes, identified by the ranking table csv
PROVIDER_CODES = LARGE_ACUTE_PROVIDER_CODES

# Get the C_999 meta-treatment code which aggregates all other treatment codes
TREATMENT_CODES = TOTAL_ONLY_TREATMENT_CODES
# TREATMENT_CODES = ALL_TREATMENT_CODES

consolidated_df = nhs.get_consolidated_df(
    start_period,
    end_period,
    PROVIDER_CODES,
    TREATMENT_CODES,
)


In [58]:
#| echo: false
#| output: false

other_trusts = consolidated_df.query("provider != 'RAJ'")

other_trusts

,period,provider,treatment,untreated,new_periods,incomplete,incomplete_prev,incomplete_diff,completed,treated,provider_name,provider_type,provider_subtype,total_treatable,incomplete_expected
0,2024-01-01,R0B,C_999,-2871,18812,60893,60550,343,15598,15598,South Tyneside and Sunderland NHS Foundation T...,Acute trust,Acute - Large,79362,63764
1,2024-02-01,R0B,C_999,-2593,18924,62414,60893,1521,14810,14810,South Tyneside and Sunderland NHS Foundation T...,Acute trust,Acute - Large,79817,65007
2,2024-03-01,R0B,C_999,-3423,17715,62324,62414,-90,14382,14382,South Tyneside and Sunderland NHS Foundation T...,Acute trust,Acute - Large,80129,65747
3,2024-04-01,R0B,C_999,-5070,18755,60982,62324,-1342,15027,15027,South Tyneside and Sunderland NHS Foundation T...,Acute trust,Acute - Large,81079,66052
4,2024-05-01,R0B,C_999,-3010,19161,61709,60982,727,15424,15424,South Tyneside and Sunderland NHS Foundation T...,Acute trust,Acute - Large,80143,64719
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
271,2024-08-01,RXR,C_999,-5467,13052,69363,72342,-2979,10564,10564,East Lancashire Hospitals NHS Trust,Acute trust,Acute - Large,85394,74830
272,2024-09-01,RXR,C_999,-3592,13788,68142,69363,-1221,11417,11417,East Lancashire Hospitals NHS Trust,Acute trust,Acute - Large,83151,71734
273,2024-10-01,RXR,C_999,-5202,15475,66279,68142,-1863,12136,12136,East Lancashire Hospitals NHS Trust,Acute trust,Acute - Large,83617,71481
274,2024-11-01,RXR,C_999,-3425,14071,65265,66279,-1014,11660,11660,East Lancashire Hospitals NHS Trust,Acute trust,Acute - Large,80350,68690


## Establishing a Large Acute Trust baseline

As part of the NHS Oversight Framework, NHS England publishes an acute trust league table, which identifies 23 trusts classified as Large Acute Trusts. [@nhs_england_nhs_2025] The Referral to Treatment (RTT) dataset contains monthly submissions for each of these trusts [@nhs_england_referral_2025], segmented by treatment specialty and identified by treatment code. This includes the meta-code C_999, which aggregates activity across all specialties.

Each monthly RTT submission consists of three sections. Part 1 reports completed RTT pathways (clock-stops), subdivided into admitted pathways (Part 1a) and non-admitted pathways (Part 1b). In this analysis, these are collectively referred to as *completed pathways*. Part 2 reports the number of incomplete RTT pathways remaining at the end of the reporting period, referred to here as *incomplete*. Part 3 reports the number of new RTT referrals received during the reporting period, referred to here as *new pathways*.

These quantities imply a simple accounting identity. Absent any unreported changes, the number of incomplete pathways at the end of a reporting period should equal the number of incomplete pathways at the start of the period, plus new pathways, minus completed pathways. Deviations from this identity are calculated as:

$$
\text{UnreportedRemovals}_t =
\text{Incomplete}_t -
\left(
\text{Incomplete}_{t-1}
+
\text{NewPathways}_t
-
\text{Completed}_t
\right)
$$

A negative value of this residual indicates a net reduction in incomplete pathways that is not explained by reported referrals or completions in the published dataset.


### Methods

Monthly RTT reports were consolidated into a single table and aggregated by provider using the C_999 treatment meta-code. For each period, this yields totals for observed incomplete pathways, expected incomplete pathways implied by reported flows, and the absolute residual between the two. These aggregates are then used to calculate the total number of pathways available to treat and the implied percentage net loss attributable to unreported removals.

In this analysis, values drawn from the publicly released RTT CSV files are referred to as “reported” figures, reflecting their provenance as trust-submitted data published by NHS England. Where these figures enter the accounting identity, they are treated as the “observed” quantities in the statistical sense.


In [59]:
# Echo the base df before applying style sheets
#| echo: false
#| output: false

other_result: pd.DataFrame = other_trusts.groupby(['period'])[
    [
        'incomplete_prev',
        'new_periods',
        'completed',
        "total_treatable",
        'untreated',
        "incomplete_expected",
        "incomplete"
    ]].sum().reset_index()

other_result["unexplained"] = other_result["untreated"]

other_result["unexplained_pct"] = other_result["untreated"] / other_result["total_treatable"]



### Large Acute Trusts baseline (excluding MSEFT)

Table 1 summarises monthly reported incomplete pathways, expected incomplete pathways implied by reported flows, and the resulting residual for the 22 Large Acute Trusts excluding Mid and South Essex NHS Foundation Trust (MSEFT) during calendar year 2024.

In [60]:
#| echo: false
#| output: true

(other_result[
     [
         "period",
         "incomplete",
         "incomplete_expected",
         "unexplained",
         "total_treatable",
         "unexplained_pct",
     ]]
 .style.relabel_index(
    [
        "",
        'Observed<br>Incomplete<br>Pathways',
        'Expected<br>Incomplete<br>Pathways',
        'Unreported<br>Change',
        'Treatable<br>Pathways',
        'Unreported<br>%'
    ], axis=1).hide(axis='index')
 .format(precision=3, thousands=",", decimal=".")
 .format('{:.2%}', subset=["unexplained_pct"])
 .format(lambda v: v.strftime("%Y-%m"), subset=["period"])
 )

,ObservedIncompletePathways,ExpectedIncompletePathways,UnreportedChange,TreatablePathways,Unreported%
2024-01,"1,355,909","1,406,947","-51,038","1,675,204",-3.05%
2024-02,"1,356,430","1,406,080","-49,650","1,659,809",-2.99%
2024-03,"1,355,242","1,401,626","-46,384","1,649,376",-2.81%
2024-04,"1,363,802","1,408,342","-44,540","1,662,020",-2.68%
2024-05,"1,371,241","1,417,668","-46,427","1,677,729",-2.77%
2024-06,"1,377,303","1,419,740","-42,437","1,670,751",-2.54%
2024-07,"1,372,476","1,426,756","-54,280","1,704,923",-3.18%
2024-08,"1,368,441","1,415,013","-46,572","1,658,678",-2.81%
2024-09,"1,364,129","1,406,322","-42,193","1,671,750",-2.52%
2024-10,"1,359,924","1,411,835","-51,911","1,700,623",-3.05%


In [61]:
#| echo: false
#| output: true

display_other_summary_df = pd.DataFrame({
    'Metric': ['Mean', 'Total'],
    'incomplete': [other_result["incomplete"].mean(), None],
    'incomplete_expected': [other_result["incomplete_expected"].mean(), None],
    'unexplained': [other_result["unexplained"].mean(), other_result["unexplained"].sum()],
    'total_treatable': [other_result["total_treatable"].mean(), None],
    'unexplained_pct': [
        other_result['untreated'].sum() / (other_result['incomplete_prev'].sum() + other_result['new_periods'].sum()),
        None],
})

display_other_summary_gt = (
    GT(
        display_other_summary_df)
    .fmt_number(columns=[
        'incomplete',
        'incomplete_expected',
        'unexplained',
        'total_treatable',
    ], decimals=0)
    .fmt_percent(columns=['unexplained_pct'], decimals=2)
    .cols_label(
        Metric=html("<nbsp/>"),
        incomplete=html("<center>Observed<br>Incomplete<br>Pathways</center>"),
        incomplete_expected=html("Expected<br>Incomplete<br>Pathways"),
        unexplained=html("Unreported<br>Change"),
        total_treatable=html('Treatable<br>Pathways'),
        unexplained_pct=html("Unreported<br>%"),
    )
)

display_other_summary_gt

,ObservedIncompletePathways,ExpectedIncompletePathways,UnreportedChange,TreatablePathways,Unreported%
Mean,"1,360,846","1,408,053","−47,207","1,668,495",−2.83%
Total,,,"−566,487",,



Across this group, the mean unreported net reduction in incomplete pathways was approximately 2.83% per month. This figure is consistent with the approximately 3% unreported removal rate previously reported across all provider types for the period April 2023 to March 2025 [@watson_why_2025].

In absolute terms, these 22 Large Acute Trusts collectively recorded approximately 566,000 unreported pathway removals during 2024.


In [62]:
#| echo: false
#| output: false

raj = consolidated_df.query("provider == 'RAJ'")

raj_result: pd.DataFrame = raj.groupby(['period'])[
    [
        'incomplete_prev',
        'new_periods',
        'completed',
        "total_treatable",
        'untreated',
        "incomplete_expected",
        "incomplete",
    ]].sum().reset_index()

raj_result["unexplained"] = raj_result["untreated"]
raj_result["unexplained_pct"] = raj_result["untreated"] / raj_result["total_treatable"]

raj_result[["period", "unexplained_pct", "incomplete", "unexplained", "total_treatable", "unexplained_pct"]]


,period,unexplained_pct,incomplete,unexplained,total_treatable,unexplained_pct
0,2024-01-01,-0.094100,160406,-19284,204932,-0.094100
1,2024-02-01,-0.078927,163701,-16091,203871,-0.078927
2,2024-03-01,-0.101640,164111,-21245,209022,-0.101640
3,2024-04-01,-0.075691,164343,-15461,204266,-0.075691
4,2024-05-01,-0.085040,165697,-17804,209360,-0.085040
5,2024-06-01,-0.077418,168084,-16160,208737,-0.077418
6,2024-07-01,-0.106382,163119,-22639,212808,-0.106382
7,2024-08-01,-0.060861,169019,-12437,204352,-0.060861
8,2024-09-01,-0.087315,166363,-18304,209632,-0.087315
9,2024-10-01,-0.094658,164086,-19801,209184,-0.094658


## Mid and South Essex NHS Foundation Trust

Applying the same methodology to Mid and South Essex NHS Foundation Trust over the same reporting period produces materially different results, as shown in Table 3.


In [63]:
#| echo: false
#| output: true

(raj_result[
    [
        "period",
        "incomplete",
        "incomplete_expected",
        "unexplained",
        "total_treatable",
        "unexplained_pct",
    ]].style.relabel_index(
    [
        "",
        'Observed<br>Incomplete<br>Pathways',
        'Expected<br>Incomplete<br>Pathways',
        'Unreported<br>Change',
        'Treatable<br>Pathways',
        'Unreported<br>%'
    ], axis=1).hide(axis='index')
.format(precision=3, thousands=",", decimal=".")
.format('{:.2%}', subset=["unexplained_pct"])
.format(lambda v: v.strftime("%Y-%m"), subset=["period"])
.set_table_styles([
    {'selector': 'th.col_heading', 'props': 'text-align: center;'},
    # {'selector': 'th.col_heading.level0', 'props': 'font-size: 1.5em;'},
    # {'selector': 'td', 'props': 'text-align: center; font-weight: bold;'},
], overwrite=False)
)

,ObservedIncompletePathways,ExpectedIncompletePathways,UnreportedChange,TreatablePathways,Unreported%
2024-01,"160,406","179,690","-19,284","204,932",-9.41%
2024-02,"163,701","179,792","-16,091","203,871",-7.89%
2024-03,"164,111","185,356","-21,245","209,022",-10.16%
2024-04,"164,343","179,804","-15,461","204,266",-7.57%
2024-05,"165,697","183,501","-17,804","209,360",-8.50%
2024-06,"168,084","184,244","-16,160","208,737",-7.74%
2024-07,"163,119","185,758","-22,639","212,808",-10.64%
2024-08,"169,019","181,456","-12,437","204,352",-6.09%
2024-09,"166,363","184,667","-18,304","209,632",-8.73%
2024-10,"164,086","183,887","-19,801","209,184",-9.47%


In [64]:
#| echo: false
#| output: true

display_mseft_summary_df = pd.DataFrame({
    'Metric': ['Mean', 'Total'],
    'incomplete': [raj_result["incomplete"].mean(), None],
    'incomplete_expected': [raj_result["incomplete_expected"].mean(), None],
    'unexplained': [raj_result["unexplained"].mean(), raj_result["unexplained"].sum()],
    'total_treatable': [raj_result["total_treatable"].mean(), None],
    'unexplained_pct': [raj['untreated'].sum() / (raj['incomplete_prev'].sum() + raj['new_periods'].sum()), None],
})

display_mseft_summary_gt = (
    GT(
        display_mseft_summary_df)
    .fmt_number(columns=['incomplete', 'incomplete_expected', 'unexplained', 'total_treatable'], decimals=0)
    .fmt_percent(columns=['unexplained_pct'], decimals=2)
    .cols_label(
        Metric=html("<nbsp/>"),
        incomplete=html("<center>Observed<br>Incomplete<br>Pathways</center>"),
        incomplete_expected=html("Expected<br>Incomplete<br>Pathways"),
        unexplained=html("Unreported<br>Change"),
        total_treatable=html('Treatable<br>Pathways'),
        unexplained_pct=html("Unreported<br>%"),
    )
)

display_mseft_summary_gt

,ObservedIncompletePathways,ExpectedIncompletePathways,UnreportedChange,TreatablePathways,Unreported%
Mean,"164,670","182,621","−17,951","207,024",−8.67%
Total,,,"−215,408",,


Across 2024, MSEFT exhibits a mean unreported removal rate of 8.67% of incomplete pathways per month—approximately three times the Large Acute Trust baseline (8.67% vs 2.83%). In absolute terms, this corresponds to 215,408 unreported pathway removals during the year.

Although MSEFT represents only one of the 23 Large Acute Trusts included in this analysis, its unreported removals account for approximately 27.5% of the total unreported removals observed across all Large Acute Trusts in 2024.


## Excess unreported removals

To control for national effects such as validation exercises, seasonal pressures, or system-wide disruptions (e.g. winter capacity constraints), a period-by-period comparison was conducted between MSEFT and the Large Acute Trust baseline excluding MSEFT.

For each month, the excess unreported removal rate was calculated as the difference between MSEFT’s unreported percentage loss and the corresponding baseline percentage loss among other Large Acute Trusts. Applying this differential to MSEFT’s total pathways yields an estimate of excess unreported removals attributable to deviation from the baseline.



In [65]:
#| echo: false
#| output: false

comparison_df = (
    raj_result.add_prefix("raj_")
    .merge(other_result.add_prefix("other_"), left_on="raj_period", right_on="other_period")
)
comparison_df

,raj_period,raj_incomplete_prev,raj_new_periods,raj_completed,raj_total_treatable,raj_untreated,raj_incomplete_expected,raj_incomplete,raj_unexplained,raj_unexplained_pct,other_period,other_incomplete_prev,other_new_periods,other_completed,other_total_treatable,other_untreated,other_incomplete_expected,other_incomplete,other_unexplained,other_unexplained_pct
0,2024-01-01,160359,44573,25242,204932,-19284,179690,160406,-19284,-0.094100,2024-01-01,1362127,313077,268257,1675204,-51038,1406947,1355909,-51038,-0.030467
1,2024-02-01,160406,43465,24079,203871,-16091,179792,163701,-16091,-0.078927,2024-02-01,1355909,303900,253729,1659809,-49650,1406080,1356430,-49650,-0.029913
2,2024-03-01,163701,45321,23666,209022,-21245,185356,164111,-21245,-0.101640,2024-03-01,1356430,292946,247750,1649376,-46384,1401626,1355242,-46384,-0.028122
3,2024-04-01,164111,40155,24462,204266,-15461,179804,164343,-15461,-0.075691,2024-04-01,1355242,306778,253678,1662020,-44540,1408342,1363802,-44540,-0.026799
4,2024-05-01,164343,45017,25859,209360,-17804,183501,165697,-17804,-0.085040,2024-05-01,1363802,313927,260061,1677729,-46427,1417668,1371241,-46427,-0.027673
5,2024-06-01,165697,43040,24493,208737,-16160,184244,168084,-16160,-0.077418,2024-06-01,1371241,299510,251011,1670751,-42437,1419740,1377303,-42437,-0.025400
6,2024-07-01,168084,44724,27050,212808,-22639,185758,163119,-22639,-0.106382,2024-07-01,1377303,327620,278167,1704923,-54280,1426756,1372476,-54280,-0.031837
7,2024-08-01,163119,41233,22896,204352,-12437,181456,169019,-12437,-0.060861,2024-08-01,1372476,286202,243665,1658678,-46572,1415013,1368441,-46572,-0.028078
8,2024-09-01,169019,40613,24965,209632,-18304,184667,166363,-18304,-0.087315,2024-09-01,1368441,303309,265428,1671750,-42193,1406322,1364129,-42193,-0.025239
9,2024-10-01,166363,42821,25297,209184,-19801,183887,164086,-19801,-0.094658,2024-10-01,1364129,336494,288788,1700623,-51911,1411835,1359924,-51911,-0.030525


In [66]:
#| echo: false
#| output: false

comparison_df["excess_pct"] = comparison_df["raj_unexplained_pct"] - comparison_df["other_unexplained_pct"]
comparison_df["excess_unexplained"] = comparison_df["excess_pct"] * comparison_df["raj_incomplete_expected"]

comparison_df[[
    "raj_period",
    "raj_unexplained",
    "raj_unexplained_pct",
    "other_unexplained_pct",
    "excess_pct",
    "excess_unexplained"
]]


,raj_period,raj_unexplained,raj_unexplained_pct,other_unexplained_pct,excess_pct,excess_unexplained
0,2024-01-01,-19284,-0.094100,-0.030467,-0.063633,-11434.172260
1,2024-02-01,-16091,-0.078927,-0.029913,-0.049014,-8812.375532
2,2024-03-01,-21245,-0.101640,-0.028122,-0.073518,-13626.978000
3,2024-04-01,-15461,-0.075691,-0.026799,-0.048892,-8790.942352
4,2024-05-01,-17804,-0.085040,-0.027673,-0.057368,-10527.011216
5,2024-06-01,-16160,-0.077418,-0.025400,-0.052018,-9584.011766
6,2024-07-01,-22639,-0.106382,-0.031837,-0.074545,-13847.341985
7,2024-08-01,-12437,-0.060861,-0.028078,-0.032783,-5948.651971
8,2024-09-01,-18304,-0.087315,-0.025239,-0.062076,-11463.405711
9,2024-10-01,-19801,-0.094658,-0.030525,-0.064134,-11793.334442


In [67]:
#| echo: false
#| output: true

summary_display_df = comparison_df[[
    "raj_period",
    "raj_unexplained",
    "raj_unexplained_pct",
    "other_unexplained_pct",
    "excess_pct",
    "excess_unexplained"
]]

# Style it
comparison_styled = (
    summary_display_df
    .style
    .hide(axis='index')
    .relabel_index(
        [
            "Period",
            'MSEFT<br>Unreported<br>Change',
            'MSEFT<br>Unreported<br>%',
            'Other<br>Unreported<br>%',
            'Excess<br>Unreported<br>%',
            'Excess <br>Unreported<br>Loss'
        ], axis=1).hide(axis='index')
    .format(
        {
            'raj_period': lambda x: x.strftime('%Y-%m') if isinstance(x, pd.Timestamp) else x,
            'raj_unexplained': '{:,.0f}',
            'raj_unexplained_pct': '{:,.2%}',
            'other_unexplained_pct': '{:,.2%}',
            'excess_pct': '{:,.2%}',
            'excess_unexplained': '{:<,.2f}',
        })
)

# comparison_styled = comparison_styled.set_table_styles([
#     {'selector': 'th.col_heading', 'props': 'text-align: center;'},
#     {'selector': 'th', 'props': 'text-align: center;'},
#     {'selector': 'td', 'props': 'text-align: right; font-weight: bold;'},
# ])

comparison_styled


Period,MSEFTUnreportedChange,MSEFTUnreported%,OtherUnreported%,ExcessUnreported%,Excess UnreportedLoss
2024-01,"-19,284",-9.41%,-3.05%,-6.36%,"-11,434.17"
2024-02,"-16,091",-7.89%,-2.99%,-4.90%,"-8,812.38"
2024-03,"-21,245",-10.16%,-2.81%,-7.35%,"-13,626.98"
2024-04,"-15,461",-7.57%,-2.68%,-4.89%,"-8,790.94"
2024-05,"-17,804",-8.50%,-2.77%,-5.74%,"-10,527.01"
2024-06,"-16,160",-7.74%,-2.54%,-5.20%,"-9,584.01"
2024-07,"-22,639",-10.64%,-3.18%,-7.45%,"-13,847.34"
2024-08,"-12,437",-6.09%,-2.81%,-3.28%,"-5,948.65"
2024-09,"-18,304",-8.73%,-2.52%,-6.21%,"-11,463.41"
2024-10,"-19,801",-9.47%,-3.05%,-6.41%,"-11,793.33"


In [68]:
#| echo: false
#| output: true

display_comparison_summary_df = pd.DataFrame(
    {
        'Metric': ['Mean', 'Total'],
        'raj_unexplained': [comparison_df["raj_unexplained"].mean(), comparison_df['raj_unexplained'].sum()],
        'raj_unexplained_pct': [comparison_df["raj_unexplained_pct"].mean(), None],
        'other_unexplained_pct': [comparison_df["other_unexplained_pct"].mean(), None],
        'excess_pct': [comparison_df["excess_pct"].mean(), None],
        'excess_unexplained': [comparison_df['excess_unexplained'].mean(), comparison_df['excess_unexplained'].sum()],
    })

display_comparison_summary_gt = (
    GT(
        display_comparison_summary_df)
    .fmt_number(columns=['raj_unexplained', 'excess_unexplained'], decimals=0)
    .fmt_percent(columns=['raj_unexplained_pct', 'other_unexplained_pct', 'excess_pct'],
                 decimals=2)
    .cols_label(
        Metric=html("<nbsp/>"),
        raj_unexplained=html("MSEFT<br>Unreported<br>Change"),
        raj_unexplained_pct=html("MSEFT<br>Unreported<br>%"),
        other_unexplained_pct=html("Other<br>Unreported<br>%"),
        excess_pct=html('Excess<br>Unreported<br>%'),
        excess_unexplained=html("Excess<br>Unreported<br>Loss"),
    )
)

display_comparison_summary_gt

,MSEFTUnreportedChange,MSEFTUnreported%,OtherUnreported%,ExcessUnreported%,ExcessUnreportedLoss
Mean,"−17,951",−8.66%,−2.83%,−5.83%,"−10,664"
Total,"−215,408",,,,"−127,974"



Under this comparison, MSEFT recorded 127,974 unreported pathway removals in excess of what would be expected given the observed Large Acute Trust baseline in 2024.

## Interpretation and implications

Individual RTT pathway removals may be clinically, administratively, or operationally justified, and this analysis does not assess the validity of individual clock-stop decisions. However, the scale, persistence, and concentration of unreported removals observed at MSEFT—particularly when compared with otherwise similar Large Acute Trusts over the same period—raise questions about the consistency and transparency of RTT waiting list management practices.

Further analysis is planned to examine pathway-level mechanisms and alternative explanatory hypotheses.

At this scale, even a small proportion of affected pathways corresponding to unresolved or abandoned care would translate into significant clinical risk for patients who were referred precisely because investigation or treatment was deemed necessary. Nonetheless, the magnitude of the discrepancies identified here is sufficient, even at this preliminary stage, to warrant closer scrutiny.



## References